In [2]:
import ollama
from ollama import chat
import sqlite3

In [3]:
def query_product_database(sql_query: str) -> str:
    """Execute a SQL query on the product.db database and return the results.
    
    Args:
        sql_query: A valid SQLite SELECT query to run against the database.
        
    Returns:
        The rows returned from the database query as a string.
    """
    try:
        # Connecting to the database
        conn = sqlite3.connect('product.db')
        cursor = conn.cursor()
        
        # Executing the query requested by the LLM
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        
        conn.close()
        return str(rows)
    except Exception as e:
        return f"Error executing query: {str(e)}"

In [ ]:
# Initialize your chat messages with a question about your database
messages = [
    {
        'role': 'user', 
        'content': 'Which product has been sold the most based on the database?'
    }
]

while True:
    response = chat(
        model='qwen3:8b',
        messages=messages,
        tools=[query_product_database], # <--- Passing your database tool here!
        think=True,
    )
    
    # Append the assistant's response to history
    messages.append(response.message)
    
    # Print out what the LLM is thinking or saying
    if response.message.thinking:
        print("Thinking:", response.message.thinking)
    if response.message.content:
        print("Answer:", response.message.content)
        
    # Check if the LLM wants to use our database function
    if not response.message.tool_calls:
        break # Exit the loop if it has its final answer
        
    # If the AI requested a tool call, execute it
    for call in response.message.tool_calls:
        if call.function.name == 'query_product_database':
            # Extract the SQL query the AI decided to write
            print(f"\n[AI generated SQL]: {call.function.arguments['sql_query']}")
            
            # Run your Python function with that SQL query
            result = query_product_database(**call.function.arguments)
            print(f"[Database Result]: {result}\n")
        else:
            result = 'Unknown tool'
            
        # Feed the database results back into the chat history
        messages.append({
            'role': 'tool', 
            'tool_name': call.function.name, 
            'content': result
        })

Thinking: Okay, the user is asking which product has been sold the most based on the database. Let me think about how to approach this.

First, I need to figure out what tables and data are available in the product.db database. Since the user mentioned a product database, there might be tables like products, sales, or orders. But without knowing the exact schema, I have to make some assumptions.

Assuming there's a table that tracks sales, maybe named 'sales' or 'order_details', which includes product IDs and quantities sold. Alternatively, there could be a 'products' table with product IDs and names, and a 'sales' table with product IDs and quantities. To find the most sold product, I'd need to aggregate the quantities sold per product.

So, the plan is to write a SQL query that sums the quantities sold for each product and then orders them in descending order to get the top one. If there's a 'sales' table with columns like product_id and quantity, the query would look something like: